<a href="https://colab.research.google.com/github/Adylbekovab/NN_Project/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Setup phase

In [1]:
#Installing Github
!apt install git

#Installing Libraries
!pip install tensorflow torch torchvision opencv-python nltk transformers


!pip freeze > requirements.txt


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.12).
0 upgraded, 0 newly installed, 0 to remove and 20 not upgraded.


## Libraries installation tests

In [37]:
import tensorflow as tf
import torch
import cv2
import nltk
from transformers import pipeline

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("OpenCV version:", cv2.__version__)

# Checking nltk
#nltk.download('punkt')

# Checking the Transformers
 #nlp = pipeline("sentiment-analysis")
#result = nlp("Image to Text")
#print(result)


try:
    nltk.download("punkt")  # Download tokenizer for NLP
    nlp = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
    result = nlp("Image to Text")
    print("✅ Transformers test successful:", result)
except Exception as e:
    print("⚠ Error checking Transformers:", e)

TensorFlow version: 2.18.0
PyTorch version: 2.5.1+cu124
OpenCV version: 4.11.0


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Device set to use cpu


✅ Transformers test successful: [{'label': 'POSITIVE', 'score': 0.993118166923523}]


### Working with Kaggle

In [11]:
import shutil
import os

# Create .kaggle directory if it does not exist
os.makedirs("/root/.kaggle", exist_ok=True)

# Move the kaggle.json file
shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")

# Set correct permissions
os.chmod("/root/.kaggle/kaggle.json", 600)

print("✅ kaggle.json successfully moved and configured!")


✅ kaggle.json successfully moved and configured!


### Verify Kaggle API Access

In [5]:
!pip install kaggle --quiet
!kaggle datasets list


ref                                                               title                                              size  lastUpdated          downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
asinow/car-price-dataset                                          Car Price Dataset                                 135KB  2025-01-26 19:53:28          12939        189  1.0              
adilshamim8/sleep-cycle-and-productivity                          Sleep Cycle & Productivity                        155KB  2025-02-07 05:44:59           1736         34  1.0              
krishnanshverma/imdb-movies-dataset                               IMDb Movies Dataset                                38KB  2025-02-12 17:53:40            881         27  1.0              
mzohaibzeeshan/google-stock-price-data-2020-2025-googl      

### Redownload the dataset

In [7]:
!kaggle datasets download -d hsankesara/flickr-image-dataset --unzip -p ./flickr30k


Dataset URL: https://www.kaggle.com/datasets/hsankesara/flickr-image-dataset
License(s): CC0-1.0
100% 8.16G/8.16G [02:16<00:00, 140MB/s]
100% 8.16G/8.16G [02:16<00:00, 64.4MB/s]


## Data Selection and Processing for Flickr30k

In [31]:
import pandas as pd
import os
import json

# Define dataset path
#dataset_path = "./flickr30k"
json_path = os.path.join(dataset_path, "flickr30k", "dataset.json")

# Load captions from the correct annotation file
annotations_file = os.path.join(dataset_path, "annotations", "captions.txt")
captions_dict = {}

# Check if the file exists
if not os.path.exists(annotations_file):
    raise FileNotFoundError(f"The file {annotations_file} was not found. Please check the path.")

# Read captions file and store captions per image
with open(annotations_file, "r") as file:
    for line in file:
        image_id, caption = line.strip().split("\t")
        image_id = image_id.split("#")[0]  # Remove caption number (#0, #1, #2...)
        if image_id not in captions_dict:
            captions_dict[image_id] = []
        captions_dict[image_id].append(caption)

# Convert to DataFrame
df = pd.DataFrame(list(captions_dict.items()), columns=["image", "captions"])
print(df.head())


            image                                           captions
0  1000092795.jpg  [Two young guys with shaggy hair look at their...
1    10002456.jpg  [Several men in hard hats are operating a gian...
2  1000268201.jpg  [A child in a pink dress is climbing up a set ...
3  1000344755.jpg  [Someone in a blue shirt and hat is standing o...
4  1000366164.jpg  [Two men, one in a gray shirt, one in a black ...


In [34]:
# Check if the dataset annotations exist, otherwise download them
if not os.path.exists(json_path):
    print("🔄 Downloading annotation files...")
    !wget -q https://cs.stanford.edu/people/karpathy/deepimagesent/flickr30k.zip
    !unzip -q flickr30k.zip -d {dataset_path}
    print("✅ Annotations downloaded!")

### Check dataset for Captions


In [28]:
import json

json_path = "./flickr30k/flickr30k/dataset.json"

# Load the JSON file
try:
    with open(json_path, "r") as file:
        data = json.load(file)
        print("✅ JSON annotations loaded successfully!")
except Exception as e:
    raise ValueError(f"⚠ Error loading JSON file: {e}")

# Print structure of JSON file
print(json.dumps(data, indent=4)[:1000])  # Show only first 1000 characters


✅ JSON annotations loaded successfully!
{
    "images": [
        {
            "sentids": [
                0,
                1,
                2,
                3,
                4
            ],
            "imgid": 0,
            "sentences": [
                {
                    "tokens": [
                        "two",
                        "young",
                        "guys",
                        "with",
                        "shaggy",
                        "hair",
                        "look",
                        "at",
                        "their",
                        "hands",
                        "while",
                        "hanging",
                        "out",
                        "in",
                        "the",
                        "yard"
                    ],
                    "raw": "Two young guys with shaggy hair look at their hands while hanging out in the yard.",
                    "imgid": 0,
                

### Extract Captions from dataset

In [24]:
import json
import os

json_path = "./flickr30k/flickr30k/dataset.json"
annotations_path = "./flickr30k/annotations"

# Create annotations directory if it doesn't exist
os.makedirs(annotations_path, exist_ok=True)

# Load JSON data
with open(json_path, "r") as file:
    data = json.load(file)

# Extract image-caption pairs
captions_file = os.path.join(annotations_path, "captions.txt")

with open(captions_file, "w") as f:
    for image in data["images"]:
        img_id = image["filename"]
        for sentence in image["sentences"]:
            caption = sentence["raw"]
            f.write(f"{img_id}\t{caption}\n")

print("✅ Captions extracted and saved as captions.txt!")


✅ Captions extracted and saved as captions.txt!


### Update Script

In [25]:
annotations_file = "./flickr30k/annotations/captions.txt"
